<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part F: Appendices</h2>
<h2>Notebook F01a: Preparing the OPS Datasets</h2>
</div>

This notebook downloads the OPS electricity consumption datasets from Hugging Face, saves them to the correct location in the repository, and runs a quick validation so you can be confident the files are ready to use.

Run it once before starting Parts A, B, or C. You do not need to run it again unless you want to refresh the data.

---

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Dataset Overview</h3>
</div>

The OPS datasets contain electricity consumption data for several European countries, sourced from the [ENTSO-E Transparency Platform](https://transparency.entsoe.eu/). Two temporal resolutions are available:

| File | Frequency | Used in |
|------|-----------|---------|
| `ops_data_15min.parquet` | 15 minutes | Parts A, B, C |
| `ops_data_30min.parquet` | 30 minutes | Parts A, B, C |

Both files are hosted in the Hugging Face repository [`mt0rm0/opsdata`](https://huggingface.co/datasets/mt0rm0/opsdata) and will be saved to `data/raw/opsdata/` inside the course repository.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Download</h3>
</div>

We use the `huggingface_hub` library to download the files. It handles authentication, caching, and resuming interrupted downloads automatically.

In [ ]:
from huggingface_hub import hf_hub_download

import nb_config
from nb_config import OPS_15M_PATH, OPS_30M_PATH

HF_REPO = "mt0rm0/opsdata"
HF_REPO_TYPE = "dataset"

files_to_download = [
    ("ops_data_15min.parquet", OPS_15M_PATH),
    ("ops_data_30min.parquet", OPS_30M_PATH),
]

for filename, destination in files_to_download:
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {filename} ...")
    hf_hub_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        filename=filename,
        local_dir=destination.parent,
    )
    print(f"  Saved to: {destination}")

print("\nDownload complete.")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Light Cleaning</h3>
</div>

We apply a small set of standardisation steps to both files before saving them back to disk:

- Parse the timestamp column and set it as the index.
- Make sure the index has a proper timezone-aware `DatetimeIndex`.
- Rename columns to lowercase with underscores for consistency with the rest of the course.
- Sort by time, just in case the source file is not already ordered.

The cleaned files overwrite the originals in `data/raw/opsdata/`, so downstream notebooks always load tidy data.

In [ ]:
import pandas as pd

def clean_ops(path: "Path") -> pd.DataFrame:
    """Load, standardise, and return an OPS dataframe."""
    df = pd.read_parquet(path)

    # Normalise column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )

    # Identify and set the datetime index
    time_candidates = [c for c in df.columns if "time" in c or "date" in c or "timestamp" in c]
    if time_candidates:
        time_col = time_candidates[0]
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df = df.set_index(time_col)
    elif not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)

    df = df.sort_index()
    return df


for _, destination in files_to_download:
    print(f"Cleaning {destination.name} ...")
    df = clean_ops(destination)
    df.to_parquet(destination)
    print(f"  Shape: {df.shape}")
    print(f"  Index: {df.index.dtype}  |  range: {df.index.min()} to {df.index.max()}")
    print()

print("Cleaning complete.")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Validation</h3>
</div>

A quick sanity check to confirm that the files are present, readable, and have the expected structure. Every check should show a green tick before you move on.

In [ ]:
import pandas as pd

def validate_ops(path, label):
    print(f"--- {label} ---")
    issues = []

    if not path.exists():
        print(f"  ❌  File not found at {path}")
        return
    print(f"  ✅  File exists")

    df = pd.read_parquet(path)
    print(f"  ✅  Readable  |  shape: {df.shape}")

    if isinstance(df.index, pd.DatetimeIndex):
        print(f"  ✅  DatetimeIndex  |  {df.index.min()} to {df.index.max()}")
    else:
        issues.append("Index is not a DatetimeIndex")
        print(f"  ❌  Index is not a DatetimeIndex (got {type(df.index).__name__})")

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if numeric_cols:
        print(f"  ✅  Numeric columns ({len(numeric_cols)}): {numeric_cols[:5]}{' ...' if len(numeric_cols) > 5 else ''}")
    else:
        issues.append("No numeric columns found")
        print(f"  ❌  No numeric columns found")

    missing_pct = df.isnull().mean().mean() * 100
    print(f"  {'✅' if missing_pct < 5 else '⚠️ '}  Missing values: {missing_pct:.2f}% overall")

    if not issues:
        print(f"  All checks passed.")
    print()


validate_ops(OPS_15M_PATH, "ops_data_15min")
validate_ops(OPS_30M_PATH, "ops_data_30min")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Quick Preview</h3>
</div>

A final look at the first few rows of each dataset to confirm everything looks right.

In [ ]:
import pandas as pd
from IPython.display import display

print("ops_data_15min.parquet")
df_15m = pd.read_parquet(OPS_15M_PATH)
display(df_15m.head())
print(f"Shape: {df_15m.shape}")

In [ ]:
print("ops_data_30min.parquet")
df_30m = pd.read_parquet(OPS_30M_PATH)
display(df_30m.head())
print(f"Shape: {df_30m.shape}")

---

The OPS datasets are ready. You can now open any notebook in Parts A, B, or C that uses them.